# Wet Oxidation Reactor 

BFB Reactor `Fe + H2O -> FeO/Fe3O4 + H2` at industrial scale

## 1. Imports 

In [1]:
# Path setup
import sys, os
_cwd = os.getcwd()
for _p in (_cwd, os.path.dirname(_cwd), os.path.dirname(os.path.dirname(_cwd))):
    if os.path.isdir(os.path.join(_p, "custom_properties")):
        sys.path.insert(0, _p)
        break

import time
from pyomo.environ import ConcreteModel, value, Var
from idaes.core import FlowsheetBlock
from idaes.core.util import scaling as iscale
from idaes.core.solvers import get_solver
import idaes.logger as idaeslog
idaeslog.getLogger("idaes").setLevel(idaeslog.WARNING)
from idaes.models_extra.gas_solid_contactors.unit_models.bubbling_fluidized_bed import (
    BubblingFluidizedBed,
)
from custom_properties.gas_phase_thermo import CustomGasPhaseParameterBlock
from custom_properties.solid_phase_thermo import CustomSolidPhaseParameterBlock
from custom_properties.oxi_wet_reactions import OxiWetReactionParameterBlock


## 2. Variables, build and fix

In [2]:
# Operating variables
n_orifice = 2500
bed_dia = 6.5        # m 
bed_height = 5       # m 
particle_dia = 1.5e-3  # m 
T_op = 1073          # K 
P_op = 1e5           # Pa 
flow_mol_gas = 300.0    # mol/s 
flow_mass_solid = 10.0  # kg/s 
porosity = 0.27      
y_H2O_in = 0.99     
y_N2_in = 0.01
w_Fe_in = 1      
w_Al2O3_in = 0.0

# Build model
m = ConcreteModel()
m.fs = FlowsheetBlock(dynamic=False)
m.fs.gas_properties = CustomGasPhaseParameterBlock()
m.fs.solid_properties = CustomSolidPhaseParameterBlock()
m.fs.wet_reactions = OxiWetReactionParameterBlock(
    solid_property_package=m.fs.solid_properties,
    gas_property_package=m.fs.gas_properties)

m.fs.BFB = BubblingFluidizedBed(
    flow_type="co_current",
    finite_elements=20,
    transformation_method="dae.finite_difference",
    gas_phase_config={"property_package": m.fs.gas_properties},
    solid_phase_config={"property_package": m.fs.solid_properties,
                        "reaction_package": m.fs.wet_reactions})

# Fix variables
m.fs.solid_properties.particle_dia.fix(particle_dia)
m.fs.solid_properties.velocity_mf.fix(0.039624)
m.fs.solid_properties.voidage_mf.fix(0.45)
m.fs.solid_properties.voidage.fix(0.50)
m.fs.BFB.number_orifice.fix(n_orifice)
m.fs.BFB.bed_diameter.fix(bed_dia)
m.fs.BFB.bed_height.fix(bed_height)
m.fs.BFB.gas_inlet.flow_mol[0].fix(flow_mol_gas)
m.fs.BFB.gas_inlet.temperature[0].fix(T_op)
m.fs.BFB.gas_inlet.pressure[0].fix(P_op)
m.fs.BFB.gas_inlet.mole_frac_comp[0, "H2O"].fix(y_H2O_in)
m.fs.BFB.gas_inlet.mole_frac_comp[0, "N2"].fix(y_N2_in)
m.fs.BFB.gas_inlet.mole_frac_comp[0, "O2"].fix(1e-5)
m.fs.BFB.gas_inlet.mole_frac_comp[0, "CO2"].fix(1e-5)
m.fs.BFB.gas_inlet.mole_frac_comp[0, "H2"].fix(1e-5)
m.fs.BFB.solid_inlet.flow_mass[0].fix(flow_mass_solid)
m.fs.BFB.solid_inlet.particle_porosity[0].fix(porosity)
m.fs.BFB.solid_inlet.temperature[0].fix(T_op)
m.fs.BFB.solid_inlet.mass_frac_comp[0, "Fe"].fix(w_Fe_in)
m.fs.BFB.solid_inlet.mass_frac_comp[0, "Al2O3"].fix(w_Al2O3_in)
m.fs.BFB.solid_inlet.mass_frac_comp[0, "Fe2O3"].fix(0)
m.fs.BFB.solid_inlet.mass_frac_comp[0, "Fe3O4"].fix(1e-5)
m.fs.BFB.solid_inlet.mass_frac_comp[0, "FeO"].fix(1e-5)

# State arguments for initializing property state blocks
gas_args = {"flow_mol": flow_mol_gas, "temperature": T_op, "pressure": P_op,
            "mole_frac": {"H2O": y_H2O_in, "N2": y_N2_in, "O2": 1e-5, "CO2": 1e-5, "H2": 1e-5}}
sol_args = {"flow_mass": flow_mass_solid, "particle_porosity": porosity, "temperature": T_op,
            "mass_frac": {"Fe": w_Fe_in, "Al2O3": w_Al2O3_in, "Fe2O3": 1e-5, "Fe3O4": 1e-5, "FeO": 1e-5}}


iscale.calculate_scaling_factors(m)

# Lower bound compositions and porosity at 0 to prevent negative values
for v in m.fs.BFB.component_data_objects(Var, descend_into=True):
    if "frac_comp" in v.name or "porosity" in v.name:
        v.setlb(0)


'fs.gas_properties.mw_comp' to mutable.
'fs.gas_properties.enth_mol_form_comp' to mutable.
'fs.gas_properties.cp_param_1' to mutable.
'fs.gas_properties.cp_param_2' to mutable.
'fs.gas_properties.cp_param_3' to mutable.
'fs.gas_properties.cp_param_4' to mutable.
'fs.gas_properties.cp_param_5' to mutable.
'fs.gas_properties.cp_param_6' to mutable.
'fs.gas_properties.cp_param_7' to mutable.
'fs.gas_properties.cp_param_8' to mutable.
'fs.solid_properties.mw_comp' to mutable.
'fs.solid_properties.dens_mass_comp_skeletal' to mutable.
'fs.solid_properties.cp_param_1' to mutable.
'fs.solid_properties.cp_param_2' to mutable.
'fs.solid_properties.cp_param_3' to mutable.
'fs.solid_properties.cp_param_4' to mutable.
'fs.solid_properties.cp_param_5' to mutable.
'fs.solid_properties.cp_param_6' to mutable.
'fs.solid_properties.cp_param_7' to mutable.
'fs.solid_properties.cp_param_8' to mutable.
'fs.solid_properties.enth_mol_form_comp' to mutable.
2026-07-25 22:57:51 [WARNING] idaes.core.util.scalin

## 3. Initialize and solve

In [3]:
#Hides the IPOPT log
m.fs.BFB.initialize(outlvl=idaeslog.CRITICAL,
                    gas_phase_state_args=gas_args,
                    solid_phase_state_args=sol_args)
solver = get_solver()
res = solver.solve(m.fs.BFB, tee=False)
terminal_condition = str(res.solver.termination_condition)
print(f"Solve: {terminal_condition}")


component keys that are not exported as part of the NL file.  Skipping.
that are not Var, Constraint, Objective, or the model.  Skipping.
component keys that are not exported as part of the NL file.  Skipping.
that are not Var, Constraint, Objective, or the model.  Skipping.
component keys that are not exported as part of the NL file.  Skipping.
that are not Var, Constraint, Objective, or the model.  Skipping.
component keys that are not exported as part of the NL file.  Skipping.
that are not Var, Constraint, Objective, or the model.  Skipping.
component keys that are not exported as part of the NL file.  Skipping.
that are not Var, Constraint, Objective, or the model.  Skipping.
component keys that are not exported as part of the NL file.  Skipping.
that are not Var, Constraint, Objective, or the model.  Skipping.
component keys that are not exported as part of the NL file.  Skipping.
that are not Var, Constraint, Objective, or the model.  Skipping.
component keys that are not export

## 4. Results and massbalance validation

In [4]:
if terminal_condition == "optimal":
  
    print(m.fs.BFB._get_stream_table_contents())

    # MW and atom counts per species

    MW = {"Fe": 0.055845, "FeO": 0.071844, "Fe3O4": 0.231533,
          "Fe2O3": 0.159688, "Al2O3": 0.101961}
    MW_gas = {"H2O": 0.018015, "H2": 0.002016, "N2": 0.028014,
              "O2": 0.031998, "CO2": 0.044009}
    n_Fe_atm = {"Fe": 1, "FeO": 1, "Fe3O4": 3, "Fe2O3": 2, "Al2O3": 0}
    n_O_atm  = {"Fe": 0, "FeO": 1, "Fe3O4": 4, "Fe2O3": 3, "Al2O3": 3}

    blk = m.fs.BFB

    # Read inlets and outlets from the solved model
    fmin_s = value(blk.solid_inlet.flow_mass[0])
    fmout_s = value(blk.solid_outlet.flow_mass[0])
    fmol_g = value(blk.gas_inlet.flow_mol[0])  
    y_in = {j: value(blk.gas_inlet.mole_frac_comp[0, j]) for j in MW_gas}
    y_out = {j: value(blk.gas_outlet.mole_frac_comp[0, j]) for j in MW_gas}
    w_in = {j: value(blk.solid_inlet.mass_frac_comp[0, j]) for j in MW}
    w_out = {j: value(blk.solid_outlet.mass_frac_comp[0, j]) for j in MW}

    # Fe atom balance 
    Fe_in = sum(w_in[j]/MW[j]*n_Fe_atm[j] for j in MW) * fmin_s
    Fe_out = sum(w_out[j]/MW[j]*n_Fe_atm[j] for j in MW) * fmout_s
    err_Fe = abs(Fe_out - Fe_in)/Fe_in*100 if Fe_in > 0 else 0

    # H2O consumed vs H2 produced 
    dH2O = (y_in["H2O"] - y_out["H2O"]) * fmol_g
    nH2 = (y_out["H2"] - y_in["H2"]) * fmol_g
    err_H = abs(nH2 - dH2O)/dH2O*100 if dH2O > 1e-10 else 0

    # O balance
    O_consumed = dH2O  
    O_solid_in = sum(w_in[j]/MW[j]*n_O_atm[j] for j in MW) * fmin_s
    O_solid_out = sum(w_out[j]/MW[j]*n_O_atm[j] for j in MW) * fmout_s
    O_incorporated = O_solid_out - O_solid_in
    err_O = abs(O_incorporated - O_consumed)/O_consumed*100 if O_consumed > 1e-10 else 0

    # Overall mass balance 
    m_gas_in = sum(y_in[j]*MW_gas[j] for j in MW_gas) * fmol_g
    m_gas_out = sum(y_out[j]*MW_gas[j] for j in MW_gas) * fmol_g
    m_total_in = m_gas_in + fmin_s
    m_total_out = m_gas_out + fmout_s
    err_mass = abs(m_total_out - m_total_in)/m_total_in*100

    # Validation
    print(f"  1. Fe atoms:  in={Fe_in:.2f}  out={Fe_out:.2f} mol_Fe/s   err={err_Fe:.2f}%")
    print(f"  2. H2O->H2:   H2O consumed={dH2O:.2f}  H2 produced={nH2:.2f} mol/s   err={err_H:.2f}%")
    print(f"  3. O balance: O consumed={O_consumed:.2f}  O in solid={O_incorporated:.2f} mol_O/s   err={err_O:.2f}%")
    print(f"  4. Mass:      in={m_total_in:.2f}  out={m_total_out:.2f} kg/s   err={err_mass:.2f}%")
    print(f"  5. Gas mass:  in={m_gas_in:.2f}  out={m_gas_out:.2f} kg/s  (d={m_gas_out-m_gas_in:+.2f})")
else:
    print("\n")


                                  Units Gas Inlet    Gas Outlet Solid Inlet  \
flow_mol                  mole / second     300.0       300.009           -   
temperature                      kelvin      1073   1147.441555        1073   
pressure                         pascal  100000.0  13296.173292           -   
mole_frac_comp N2         dimensionless      0.01          0.01           -   
mole_frac_comp O2         dimensionless   0.00001       0.00001           -   
mole_frac_comp CO2        dimensionless   0.00001       0.00001           -   
mole_frac_comp H2O        dimensionless      0.99      0.719635           -   
mole_frac_comp H2         dimensionless   0.00001      0.270345           -   
flow_mass             kilogram / second         -             -        10.0   
particle_porosity         dimensionless         -             -        0.27   
mass_frac_comp Fe2O3      dimensionless         -             -           0   
mass_frac_comp Fe3O4      dimensionless         -   